# KNN

In [4]:
import heapq
from dataclasses import dataclass
from typing import Optional, Any, List, Tuple
import numpy as np

## kdtree树类

In [ ]:
@dataclass #
class KDNode:
    """
    kd 树的节点类。
    每个节点保存：
    1. 当前样本点 point
    2. 当前样本点对应的标签 label
    3. 当前节点使用哪个坐标轴 axis 进行切分
    4. 左子树 left
    5. 右子树 right
    """
    point: np.ndarray
    label: Any
    axis: int
    left: Optional["KDNode"] = None
    right: Optional["KDNode"] = None

class KDTree:
    """
    kd 树类。

    作用：
    1. 根据训练数据 X 和标签 y 构造 kd 树；
    2. 对给定测试点，搜索最近的 k 个邻居。
    """
    def __init__(self, p: float = 2, split_strategy: str = 'cycle'):
        """
        p : float
            Lp 距离中的 p。
        split_strategy : str
            kd 树切分维度的选择方式。
            "cycle" 表示按维度循环切分；
            "variance" 表示每次选择方差最大的维度切分。
        """
        if p < 1 and p != np.inf:
            raise ValueError("p 必须满足 p >= 1,或者 p = np.inf")
        if split_strategy not in ("cycle", "variance"):
            raise ValueError("split_strategy 只能是 'cycle' 或 'variance'")
        self.p = p
        self.split_strategy = split_strategy
        self.root: Optional[KDNode] = None  #
        self.n_samples_: Optional[int] = None #
        self.n_feature_: Optional[int] = None

    def fit(self, X: np.ndarray, y: Optional[np.ndarray] = None):
        """
        构造 kd 树。
        X : np.ndarray, shape = (n_samples, n_features)
            训练样本特征矩阵。
        y : np.ndarray, shape = (n_samples,), optional
            每个样本对应的标签。
            如果 y=None，则默认标签为样本编号。
        """
        X = np.asarray(X, dtype = float)
        if X.ndim != 2:
            raise ValueError("X 必须是二维数组")
        n_samples, n_features = X.shape
        if y is None:
            y = np.arange(n_samples)
        else:
            y = np.asarray(y)

        self.n_samples = n_samples
        self.n_features = n_features

        self.root = self._build_tree(X, y, depth = 0)

        return self

    def _build_tree(self, X: np.ndarray, y: np.ndarray, depth: int) -> Optional[KDNode]:
        """
        递归构造 kd 树。
        核心思想：
        1. 选择一个切分维度 axis；
        2. 按照 axis 这一维对样本排序；
        3. 取中位数样本作为当前节点；
        4. 中位数左边的数据构造左子树；
        5. 中位数右边的数据构造右子树。
        """
        n_samples = X.shape[0]
        if n_samples == 0: #
            return None

        if self.split_strategy == 'cycle':
            axis = depth % self.n_features
        else:
            axis = int(np.argmax(np.var(X, axis = 0)))

        sorted_indices = np.argsort(X[:, axis], kind = 'mergesort') #
        X_sorted = X[sorted_indices]
        y_sorted = y[sorted_indices]
        median = n_samples // 2

        node = KDNode(
            point = X_sorted[median],
            label = y_sorted[median],
            axis = axis,
            left = self._build_tree(
                X_sorted[:median],
                y_sorted[:median],
                depth + 1
            ),
            right = self._build_tree(
                X_sorted[median + 1:],
                y_sorted[median + 1:],
                depth + 1
            )
        )
        return node

    def _distance(self, x1: np.ndarray, x2: np.ndarray) -> float:
        return float(np.linalg.norm(x1 - x2, ord = self.p))

    def query(self, point: np.ndarray, k: int = 1) -> List[Tuple[float, np.ndarray, Any]]:
        """
        查询距离 point 最近的 k 个样本点。

        参数
        ----------
        point : np.ndarray, shape = (n_features,)
            待查询样本点。

        k : int
            要搜索的最近邻个数。

        返回
        ----------
        neighbors : list
            返回一个列表，每个元素为：

            (distance, neighbor_point, neighbor_label)

            其中：
            distance 是距离；
            neighbor_point 是邻居点坐标；
            neighbor_label 是邻居点标签。
        """
        if self.root is None:
            raise RuntimeError

        point = np.asarray(point, dtype = float)
        if point.ndim != 1:
            raise ValueError("point 必须是一维数组，形状为 (n_features,)")
        if point.shape[0] != self.n_features:
            raise ValueError("point 的维度和训练数据维度不一致")
        if k <= 0:
            raise ValueError("k 必须是正整数")

        k = min(k, self.n_samples)
        # heap 中每个元素是：(-distance, counter, point, label)
        heap = []
        counter = 0

        def push_neighbor(distance:float, node: KDNode):
            """
            尝试把当前节点加入最近邻堆中。
            如果堆中元素不足 k 个，直接加入；
            如果堆中已有 k 个元素，则只有当前点比最远邻居更近时才替换。
            """
            nonlocal counter
            item = (-distance, counter, node.point, node.label)
            counter += 1

            if len(heap) < k:
                heapq.heappush(heap, item)
            else:
                current_worst_distance = -heap[0][0]
                if distance < current_worst_distance:
                    heapq.heapreplace(heap, item)

        def search(node: Optional[KDNode]):
            """
            递归搜索 kd 树。

            搜索过程：
            1. 根据目标点和当前节点在切分维度上的大小，先搜索更可能包含近邻的一侧；
            2. 更新当前节点是否属于最近邻；
            3. 判断另一侧子树是否可能存在更近的点；
            4. 如果可能，则回溯搜索另一侧。
            """
            if node is None:
                return
            axis = node.axis

            if point[axis] < node.point[axis]:
                near_branch = node.left
                far_branch = node.right
            else:
                near_branch = node.right
                far_branch = node.left

            search(near_branch)
            distance = self._distance(point, node.point)
            push_neighbor(distance, node)

            if len(heap) < k:
                worst_distance = np.inf
            else:
                worst_distance = -heap[0][0]

            plane_distance = abs(point[axis] - node.point[axis])
            if plane_distance <= worst_distance:
                search(far_branch)

        search(self.root)

        neighbors = [
            (-neg_distance, neighbor_point, neighbor_label)
            for neg_distance, _, neighbor_point, neighbor_label in heap
        ]
        neighbors.sort(key = lambda item: item[0])
        return neighbors

class KNNClassifier:
    """
    基于 kd 树实现的 KNN 分类器。
    支持：
    1. 设定 k 值；
    2. 设定 Lp 距离；
    3. 多数投票分类；
    4. 距离加权投票分类；
    5. predict；
    6. predict_proba。
    """
    def __init__(self, n_neighbors: int = 5, p: float = 2, weights: str = "uniform", split_strategy: str = "cycle"):
        """
        weights : str
            投票权重方式。
            "uniform" 表示普通多数投票；
            "distance" 表示距离越近，权重越大。
        """
        if n_neighbors <= 0:
            raise ValueError("n_neighbors 必须是正整数")

        if weights not in ("uniform", "distance"):
            raise ValueError("weights 只能是 'uniform' 或 'distance'")

        self.n_neighbors = n_neighbors
        self.p = p
        self.weights = weights
        self.split_strategy = split_strategy
        self.tree_: Optional[KDTree] = None
        self.classes_: Optional[np.ndarray] = None
        self.class_to_index: Optional[dict] = None
        self.n_features: Optional[int] = None
        self.n_samples: Optional[int] = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        """
        训练 KNN 分类器。
        注意：
        KNN 本身没有真正的参数训练过程。
        这里的 fit 主要做两件事：
        1. 保存类别信息；
        2. 构造 kd 树，用于加速最近邻搜索。
        """
        X = np.asarray(X, dtype = float)
        y = np.asarray(y)
        if X.ndim != 2:
            raise ValueError("X 必须是二维数组，形状为 (n_samples, n_features)")
        if y.ndim != 1:
            y = y.ravel()
        if X.shape[0] != y.shape[0]:
            raise ValueError("X 和 y 的样本数量不一致")
        self.n_samples, self.n_features= X.shape
        self.classes_ = np.unique(y)
        self.class_to_index_ = {
            label: index for index, label in enumerate(self.classes_)
        }
        self.tree_ = KDTree(p = self.p, split_strategy= self.split_strategy)
        self.tree_.fit(X, y)
        return self

    def _check_X_query(self, X: np.ndarray) -> np.ndarray:
        """
        检查预测输入 X 的形状。

        如果输入是单个样本，例如：

        [1.0, 2.0]

        会自动转成：

        [[1.0, 2.0]]
        """
        X = np.asarray(X, dtype=float)

        if X.ndim == 1:
            X = X.reshape(1, -1)

        if X.ndim != 2:
            raise ValueError("X 必须是一维或二维数组")

        if X.shape[1] != self.n_features:
            raise ValueError("X 的特征维度和训练数据不一致")

        return X
    def _vote(self, neighbors: List[Tuple[float, np.ndarray, Any]]) -> np.ndarray:
        """
        根据最近邻进行投票。

        返回每个类别的投票权重。
        """
        votes = np.zeros(len(self.classes_), dtype = float)
        if self.weights == "uniform":
            for distance, point, label in neighbors:
                class_index = self.class_to_index_[label]
                votes[class_index] += 1.0
        else:
            zero_distance_neighbors = [
                item for item in neighbors if item[0] == 0.0
            ]

            if len(zero_distance_neighbors) > 0:
                # 如果测试点和某些训练点完全重合，
                # 则只使用这些距离为 0 的点进行投票。
                for distance, point, label in zero_distance_neighbors:
                    class_index = self.class_to_index_[label]
                    votes[class_index] += 1.0
            else:
                for distance, point, label in neighbors:
                    class_index = self.class_to_index_[label]
                    votes[class_index] += 1.0 / distance

        return votes

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        预测样本类别。

        参数
        ----------
        X : np.ndarray
            可以是单个样本，形状为 (n_features,)；
            也可以是多个样本，形状为 (n_samples, n_features)。

        返回
        ----------
        y_pred : np.ndarray
            预测类别。
        """
        if self.tree_ is None:
            raise RuntimeError("模型还没有训练，请先调用 fit(X, y)")

        X = self._check_X_query(X)

        y_pred = []

        for point in X:
            neighbors = self.tree_.query(
                point,
                k=self.n_neighbors
            )

            votes = self._vote(neighbors)

            predicted_class_index = int(np.argmax(votes))
            predicted_class = self.classes_[predicted_class_index]

            y_pred.append(predicted_class)

        return np.asarray(y_pred)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        预测每个样本属于各类别的概率。

        概率的计算方式：
        某类的投票权重 / 所有类别的投票权重之和。
        """
        if self.tree_ is None:
            raise RuntimeError("模型还没有训练，请先调用 fit(X, y)")

        X = self._check_X_query(X)

        probabilities = []

        for point in X:
            neighbors = self.tree_.query(
                point,
                k=self.n_neighbors
            )

            votes = self._vote(neighbors)
            total = np.sum(votes)

            if total == 0:
                proba = np.ones(len(self.classes_)) / len(self.classes_)
            else:
                proba = votes / total

            probabilities.append(proba)

        return np.asarray(probabilities)

    def kneighbors(self, X: np.ndarray):
        """
        返回每个测试点的 k 个最近邻。

        返回格式：
        [
            [
                (distance, neighbor_point, neighbor_label),
                ...
            ],
            ...
        ]
        """
        if self.tree_ is None:
            raise RuntimeError("模型还没有训练，请先调用 fit(X, y)")

        X = self._check_X_query(X)

        all_neighbors = []

        for point in X:
            neighbors = self.tree_.query(
                point,
                k=self.n_neighbors
            )
            all_neighbors.append(neighbors)

        return all_neighbors

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """
        计算分类准确率。
        """
        y = np.asarray(y)
        y_pred = self.predict(X)

        return float(np.mean(y_pred == y))


In [8]:
if __name__ == "__main__":
    # =========================
    # 一个简单测试例子
    # =========================

    X_train = np.array([
        [2, 3],
        [5, 4],
        [9, 6],
        [4, 7],
        [8, 1],
        [7, 2]
    ])

    y_train = np.array([
        "A",
        "A",
        "B",
        "A",
        "B",
        "B"
    ])

    X_test = np.array([
        [3, 4],
        [8, 2],
        [6, 5]
    ])

    model = KNNClassifier(
        n_neighbors=3,
        p=2,
        weights="uniform",
        split_strategy="cycle"
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("预测类别：")
    print(y_pred)

    print("\n类别顺序：")
    print(model.classes_)

    print("\n预测概率：")
    print(model.predict_proba(X_test))

    print("\n最近邻：")
    neighbors = model.kneighbors(X_test)

    for i, item in enumerate(neighbors):
        print(f"\n测试点 {X_test[i]} 的最近邻：")
        for distance, point, label in item:
            print(f"距离 = {distance:.4f}, 邻居点 = {point}, 标签 = {label}")

预测类别：
['A' 'B' 'A']

类别顺序：
['A' 'B']

预测概率：
[[1.         0.        ]
 [0.33333333 0.66666667]
 [0.66666667 0.33333333]]

最近邻：

测试点 [3 4] 的最近邻：
距离 = 1.4142, 邻居点 = [2. 3.], 标签 = A
距离 = 2.0000, 邻居点 = [5. 4.], 标签 = A
距离 = 3.1623, 邻居点 = [4. 7.], 标签 = A

测试点 [8 2] 的最近邻：
距离 = 1.0000, 邻居点 = [8. 1.], 标签 = B
距离 = 1.0000, 邻居点 = [7. 2.], 标签 = B
距离 = 3.6056, 邻居点 = [5. 4.], 标签 = A

测试点 [6 5] 的最近邻：
距离 = 1.4142, 邻居点 = [5. 4.], 标签 = A
距离 = 2.8284, 邻居点 = [4. 7.], 标签 = A
距离 = 3.1623, 邻居点 = [7. 2.], 标签 = B
